# 06 — Run MiDaS on DIODE Validation

Batch inference using `dpt_hybrid_384` over the DIODE validation manifest.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'configs' / 'dataset_paths.yaml').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'configs' / 'dataset_paths.yaml').exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
MODEL_TYPE = 'dpt_hybrid_384'
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifests' / 'diode_val_manifest.csv'
PRED_ROOT = PROJECT_ROOT / 'outputs' / 'predictions' / 'diode'

# Set small for smoke test; None for full validation split
MAX_SAMPLES = 50

In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from src.adapters.midas_adapter import MiDaSAdapter

df = pd.read_csv(MANIFEST_PATH)
if MAX_SAMPLES:
    df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=42).reset_index(drop=True)

print(f'Running inference on {len(df)} images')
adapter = MiDaSAdapter(model_type=MODEL_TYPE)

In [ ]:
skipped = 0
for _, row in tqdm(df.iterrows(), total=len(df)):
    out_path = PRED_ROOT / row['domain'] / row['scene'] / row['scan'] / f"{row['frame_id']}.npy"
    if out_path.exists():
        skipped += 1
        continue
    out_path.parent.mkdir(parents=True, exist_ok=True)
    depth = adapter.predict(row['image_path'])
    np.save(str(out_path), depth)

print(f'Done. Skipped {skipped} already-existing predictions.')